# IE 643 Model Street - Phase 1 Reconstruction Harness

This is the single working notebook for Phase 1. The priority is reproducible reconstruction evaluation before API, UI, or deployment work.

What this notebook does:
- Builds CIFAR-style ResNet-18/34/50/101/152 models.
- Loads local CIFAR-10 checkpoints or documented torchvision ImageNet pretrained weights.
- Runs model-inversion reconstruction from classifier weights and penultimate features.
- Computes SSIM, PSNR, MSE, embedding cosine, target confidence, top-k consistency, and runtime.
- Saves a metrics table plus reconstruction/nearest-neighbor grids.

Research basis from the project plan:
- Mahendran and Vedaldi: optimize an image so its deep representation matches a target representation.
- DeepInversion: use BatchNorm running statistics from a fixed pretrained classifier as a data-free image prior.
- Plug-In Inversion: use augmentation consistency later in Phase 2 to reduce fragile hand-tuned priors.

Phase boundary: this notebook makes the baseline measurable and reproducible. Phase 2 should tune the objective and report metric improvement against the CSV produced here.

In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.models as tv_models
import torchvision.transforms as transforms
from PIL import Image
from skimage.metrics import structural_similarity as structural_similarity
from torchvision.transforms.functional import to_pil_image
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj: Any) -> None:
        print(obj)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Model_street_final":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "Model_street_final"
PRETRAINED_WEIGHTS_DIR = PROJECT_ROOT / "pre_trained_model_weights"
RESULTS_DIR = PROJECT_ROOT / "results" / "phase1_baseline"
FIGURE_DIR = RESULTS_DIR / "figures"
CACHE_DIR = RESULTS_DIR / "cache"
for directory in (RESULTS_DIR, FIGURE_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: D:\IE_643_Project
Device: cpu


C:\Users\Achintya Jha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiment Contract

The weight source is recorded in every result row. That matters because reconstruction quality depends heavily on what information is actually stored in the trained ResNet weights.

- `local_cifar10`: loads a checkpoint trained for this project, preferably from `pre_trained_model_weights/`.
- `torchvision_imagenet`: loads official torchvision ImageNet weights where compatible with the CIFAR-style architecture. Shape-mismatched layers are skipped and reported.
- `random`: runs the harness without pretrained information, useful only for smoke tests.

Default benchmark scope is intentionally small: ResNet-18, a few CIFAR-10 classes, and a few seeds. Expand only after the baseline path is verified.

In [2]:
CIFAR10_CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

TORCHVISION_IMAGENET_WEIGHTS = {
    "resnet18": ("ResNet18_Weights", "IMAGENET1K_V1"),
    "resnet34": ("ResNet34_Weights", "IMAGENET1K_V1"),
    "resnet50": ("ResNet50_Weights", "IMAGENET1K_V2"),
    "resnet101": ("ResNet101_Weights", "IMAGENET1K_V2"),
    "resnet152": ("ResNet152_Weights", "IMAGENET1K_V1"),
}

def resolve_weight_path(filename: str) -> Path:
    candidates = [
        PRETRAINED_WEIGHTS_DIR / filename,
        MODEL_DIR / filename,
        PROJECT_ROOT / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


LOCAL_CHECKPOINTS = {
    "resnet18": resolve_weight_path("resnet18_cifar10_trained.pth"),
    "resnet34": resolve_weight_path("resnet34_cifar10.pth"),
    "resnet50": resolve_weight_path("resnet50_cifar10 .pth"),
    "resnet101": resolve_weight_path("resnet101_cifar10_final_state_dict.pth"),
    "resnet152": resolve_weight_path("resnet152_cifar10.pth"),
}

MNIST_CHECKPOINTS = {
    "resnet18_best": resolve_weight_path("resnet18_mnist_best.pth"),
    "resnet18_final": resolve_weight_path("resnet18_mnist_final.pth"),
    "resnet34_final": resolve_weight_path("resnet34_mnist_final.pth"),
    "resnet152_final": resolve_weight_path("resnet152_mnist_final.pth"),
}

PRIOR_REPORTED_BASELINE = pd.DataFrame([
    {"variant": "resnet18", "SSIM": 0.0683, "cosine": 0.421, "PSNR": 7.249, "MSE": 0.188},
    {"variant": "resnet34", "SSIM": 0.1331, "cosine": 0.410, "PSNR": 7.281, "MSE": 0.187},
    {"variant": "resnet50", "SSIM": 0.0786, "cosine": 0.405, "PSNR": 6.119, "MSE": 0.244},
    {"variant": "resnet101", "SSIM": 0.0938, "cosine": 0.761, "PSNR": 6.462, "MSE": 0.225},
    {"variant": "resnet152", "SSIM": 0.0801, "cosine": 0.507, "PSNR": 6.532, "MSE": 0.222},
])

@dataclass(frozen=True)
class ExperimentConfig:
    variant: str = "resnet18"
    weight_source: str = "local_cifar10"
    checkpoint_path: Optional[str] = str(LOCAL_CHECKPOINTS["resnet18"])
    num_classes: int = 10
    image_size: int = 32
    class_ids: Tuple[int, ...] = (0, 1, 8)
    seeds: Tuple[int, ...] = (7, 21, 42)
    steps_z: int = 300
    steps_image: int = 500
    lr_z: float = 1e-2
    lr_image: float = 3e-2
    lambda_embedding: float = 1.0
    lambda_cosine: float = 0.5
    lambda_logit: float = 0.1
    lambda_tv: float = 1e-5
    lambda_l2: float = 1e-4
    lambda_bn: float = 1e-2
    reference_split: str = "train"
    reference_max_per_class: int = 500
    download_cifar10: bool = False
    top_k: int = 5
    device: str = str(DEVICE)

BASE_CONFIG = ExperimentConfig()
print(BASE_CONFIG)

ExperimentConfig(variant='resnet18', weight_source='local_cifar10', checkpoint_path='D:\\IE_643_Project\\pre_trained_model_weights\\resnet18_cifar10_trained.pth', num_classes=10, image_size=32, class_ids=(0, 1, 8), seeds=(7, 21, 42), steps_z=300, steps_image=500, lr_z=0.01, lr_image=0.03, lambda_embedding=1.0, lambda_cosine=0.5, lambda_logit=0.1, lambda_tv=1e-05, lambda_l2=0.0001, lambda_bn=0.01, reference_split='train', reference_max_per_class=500, download_cifar10=False, top_k=5, device='cpu')


## CIFAR-Style ResNet Builder

The project checkpoints use CIFAR-style ResNets: 3x3 first convolution, no initial max-pool, and a 10-class classifier head. This avoids silent shape confusion when loading project checkpoints.

In [3]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = None
        if stride != 1 or in_planes != planes * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=False)
        self.downsample = None
        if stride != 1 or in_planes != planes * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class ResNetCIFAR(nn.Module):
    def __init__(self, block: type[nn.Module], layers: Sequence[int], num_classes: int = 10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=False)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        self._initialize_weights()

    def _make_layer(self, block: type[nn.Module], planes: int, blocks: int, stride: int) -> nn.Sequential:
        layers = [block(self.in_planes, planes, stride)]
        self.in_planes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_planes, planes))
        return nn.Sequential(*layers)

    def _initialize_weights(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.constant_(module.weight, 1)
                nn.init.constant_(module.bias, 0)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.forward_features(x))


def build_resnet_cifar(variant: str, num_classes: int = 10) -> ResNetCIFAR:
    builders = {
        "resnet18": (BasicBlock, [2, 2, 2, 2]),
        "resnet34": (BasicBlock, [3, 4, 6, 3]),
        "resnet50": (Bottleneck, [3, 4, 6, 3]),
        "resnet101": (Bottleneck, [3, 4, 23, 3]),
        "resnet152": (Bottleneck, [3, 8, 36, 3]),
    }
    if variant not in builders:
        raise ValueError(f"Unsupported variant: {variant}")
    block, layers = builders[variant]
    return ResNetCIFAR(block, layers, num_classes=num_classes)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())

## Weight Loading and Provenance

Every loaded model returns a provenance dictionary. This makes later metric claims defensible: the notebook records whether the reconstruction came from project-trained weights, official ImageNet weights, or random initialization.

In [4]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def extract_state_dict(raw: Any) -> Dict[str, torch.Tensor]:
    if isinstance(raw, dict):
        for key in ("state_dict", "model"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]
                break
    if not isinstance(raw, dict):
        raise TypeError("Checkpoint does not contain a state_dict-like mapping.")
    return {str(key).replace("module.", ""): value for key, value in raw.items() if torch.is_tensor(value)}


def load_weights_safely(model: nn.Module, checkpoint_path: Any, map_location: str = "cpu") -> Dict[str, Any]:
    """Mirror the project script's safe loader while returning provenance for metrics."""
    raw = torch.load(checkpoint_path, map_location=map_location) if isinstance(checkpoint_path, (str, Path)) else checkpoint_path
    state_dict = extract_state_dict(raw)
    model_state = model.state_dict()
    compatible = {
        key: value for key, value in state_dict.items()
        if key in model_state and tuple(value.shape) == tuple(model_state[key].shape)
    }
    skipped_layers = [key for key in model_state if key not in compatible]
    unexpected_layers = [key for key in state_dict if key not in model_state]
    model.load_state_dict(compatible, strict=False)
    print(f"Loaded {len(compatible)}/{len(model_state)} layers from checkpoint")
    if skipped_layers:
        print("Skipped layers:", skipped_layers)
    return {
        "loaded_layers": len(compatible),
        "model_layers": len(model_state),
        "skipped_layers": skipped_layers,
        "unexpected_layers": unexpected_layers[:25],
        "load_style": "project_safe_compatible_layers",
    }



def resolve_torchvision_weights(variant: str):
    enum_name, member_name = TORCHVISION_IMAGENET_WEIGHTS[variant]
    enum = getattr(tv_models, enum_name)
    return getattr(enum, member_name), f"torchvision.{enum_name}.{member_name}"


def load_model(config: ExperimentConfig, seed: int) -> Tuple[nn.Module, Dict[str, Any]]:
    set_seed(seed)
    model = build_resnet_cifar(config.variant, config.num_classes).to(config.device)
    provenance: Dict[str, Any] = {
        "variant": config.variant,
        "weight_source": config.weight_source,
        "checkpoint_path": config.checkpoint_path,
        "parameter_count": count_parameters(model),
        "loaded_layers": 0,
        "model_layers": len(model.state_dict()),
        "skipped_layers_count": len(model.state_dict()),
        "pretrained_description": "random initialization only",
    }

    if config.weight_source == "local_cifar10":
        checkpoint_path = Path(config.checkpoint_path or LOCAL_CHECKPOINTS[config.variant])
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")
        load_report = load_weights_safely(model, checkpoint_path)
        provenance.update(load_report)
        provenance["pretrained_description"] = f"local CIFAR-10 checkpoint: {checkpoint_path.name}"
    elif config.weight_source == "torchvision_imagenet":
        weights, weight_name = resolve_torchvision_weights(config.variant)
        tv_model = getattr(tv_models, config.variant)(weights=weights)
        load_report = load_weights_safely(model, tv_model.state_dict())
        provenance.update(load_report)
        provenance["pretrained_description"] = f"official ImageNet pretrained weights: {weight_name}"
    elif config.weight_source == "random":
        pass
    else:
        raise ValueError(f"Unsupported weight_source: {config.weight_source}")

    provenance["skipped_layers_count"] = len(provenance.get("skipped_layers", []))
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model, provenance


model_smoke, provenance_smoke = load_model(replace(BASE_CONFIG, weight_source="random", checkpoint_path=None), seed=0)
print(json.dumps({key: provenance_smoke[key] for key in ("variant", "weight_source", "parameter_count", "pretrained_description")}, indent=2))
del model_smoke

{
  "variant": "resnet18",
  "weight_source": "random",
  "parameter_count": 11173962,
  "pretrained_description": "random initialization only"
}


## Reconstruction Objective

The baseline uses the project direction already present in the old notebook: classifier-weight prototype, optimized penultimate embedding, image optimization, cosine/embedding/logit terms, total variation, L2 image prior, and BatchNorm-stat alignment. The difference is that the pieces are reusable and logged instead of being hard-coded cell fragments.

In [5]:
def estimate_embedding_norm(model: ResNetCIFAR, image_size: int, device: str, samples: int = 8) -> float:
    norms: List[float] = []
    with torch.no_grad():
        for _ in range(samples):
            image = torch.rand(1, 3, image_size, image_size, device=device)
            embedding = model.forward_features(image)
            norms.append(float(embedding.norm(dim=1).mean().item()))
    return float(np.mean(norms))


def optimize_target_embedding(model: ResNetCIFAR, class_id: int, config: ExperimentConfig) -> torch.Tensor:
    fc_weight = model.fc.weight.detach().to(config.device)
    fc_bias = model.fc.bias.detach().to(config.device) if model.fc.bias is not None else None
    target_norm = estimate_embedding_norm(model, config.image_size, config.device)
    z = fc_weight[class_id].detach().clone()
    z = z / (z.norm() + 1e-8) * target_norm
    z.requires_grad_(True)
    optimizer = optim.Adam([z], lr=config.lr_z)

    for _ in range(config.steps_z):
        optimizer.zero_grad(set_to_none=True)
        logits = fc_weight @ z
        if fc_bias is not None:
            logits = logits + fc_bias
        logit_loss = -logits[class_id]
        norm_loss = (z.norm() - target_norm).pow(2)
        l2_loss = z.pow(2).mean()
        loss = logit_loss + 1e-1 * norm_loss + 1e-4 * l2_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_([z], 1.0)
        optimizer.step()
    return z.detach()


def total_variation(image: torch.Tensor) -> torch.Tensor:
    return (
        (image[:, :, 1:, :] - image[:, :, :-1, :]).pow(2).mean()
        + (image[:, :, :, 1:] - image[:, :, :, :-1]).pow(2).mean()
    )


def forward_with_bn_loss(model: ResNetCIFAR, image: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    bn_losses: List[torch.Tensor] = []
    handles = []

    def make_hook(module: nn.BatchNorm2d):
        def hook(_: nn.Module, __: Tuple[torch.Tensor, ...], output: torch.Tensor) -> None:
            if output.ndim != 4:
                return
            mean = output.mean(dim=(0, 2, 3))
            var = output.var(dim=(0, 2, 3), unbiased=False)
            running_mean = module.running_mean.to(output.device)
            running_var = module.running_var.to(output.device)
            if mean.shape == running_mean.shape:
                bn_losses.append(F.mse_loss(mean, running_mean) + F.mse_loss(var, running_var))
        return hook

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            handles.append(module.register_forward_hook(make_hook(module)))
    try:
        embedding = model.forward_features(image)
        logits = model.fc(embedding)
    finally:
        for handle in handles:
            handle.remove()
    if bn_losses:
        bn_loss = torch.stack(bn_losses).mean()
    else:
        bn_loss = torch.zeros((), device=image.device)
    return embedding, logits, bn_loss


def reconstruct_from_weights(model: ResNetCIFAR, class_id: int, seed: int, config: ExperimentConfig) -> Tuple[torch.Tensor, Dict[str, float]]:
    set_seed(seed)
    z_target = optimize_target_embedding(model, class_id, config)
    image = (torch.rand(1, 3, config.image_size, config.image_size, device=config.device) * 0.1).requires_grad_(True)
    optimizer = optim.Adam([image], lr=config.lr_image)
    start = time.perf_counter()
    last_metrics: Dict[str, float] = {}

    for _ in range(config.steps_image):
        optimizer.zero_grad(set_to_none=True)
        clipped = image.clamp(0, 1)
        embedding, logits, bn_loss = forward_with_bn_loss(model, clipped)
        embedding_loss = F.mse_loss(embedding, z_target.unsqueeze(0))
        cosine = F.cosine_similarity(embedding, z_target.unsqueeze(0), dim=1).mean()
        cosine_loss = 1.0 - cosine
        logit_loss = -logits[:, class_id].mean()
        tv_loss = total_variation(clipped)
        l2_loss = clipped.pow(2).mean()
        loss = (
            config.lambda_embedding * embedding_loss
            + config.lambda_cosine * cosine_loss
            + config.lambda_logit * logit_loss
            + config.lambda_tv * tv_loss
            + config.lambda_l2 * l2_loss
            + config.lambda_bn * bn_loss
        )
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            image.clamp_(0, 1)
        last_metrics = {
            "objective_loss": float(loss.detach().cpu().item()),
            "embedding_loss": float(embedding_loss.detach().cpu().item()),
            "embedding_cosine_to_target": float(cosine.detach().cpu().item()),
            "target_logit": float(logits[:, class_id].mean().detach().cpu().item()),
            "bn_loss": float(bn_loss.detach().cpu().item()),
        }

    with torch.no_grad():
        final_image = image.detach().clamp(0, 1)
        _, logits, _ = forward_with_bn_loss(model, final_image)
        probabilities = torch.softmax(logits, dim=1)
        top_values, top_indices = torch.topk(probabilities, k=min(config.top_k, config.num_classes), dim=1)
    last_metrics.update({
        "runtime_seconds": float(time.perf_counter() - start),
        "target_confidence": float(probabilities[0, class_id].detach().cpu().item()),
        "top1_class": int(top_indices[0, 0].detach().cpu().item()),
        "top1_confidence": float(top_values[0, 0].detach().cpu().item()),
        "topk_contains_target": bool(class_id in top_indices[0].detach().cpu().tolist()),
    })
    return final_image, last_metrics

## Dataset References and Metrics

Metrics are computed against the nearest same-class CIFAR-10 reference image. This preserves the earlier project metric style while making the class filter explicit.

In [6]:
def load_cifar10_reference_tensors(config: ExperimentConfig, class_id: int) -> torch.Tensor:
    split_is_train = config.reference_split.lower() == "train"
    dataset = torchvision.datasets.CIFAR10(
        root=str(PROJECT_ROOT / "data"),
        train=split_is_train,
        download=config.download_cifar10,
        transform=transforms.ToTensor(),
    )
    images: List[torch.Tensor] = []
    for image, label in dataset:
        if int(label) == int(class_id):
            images.append(image)
            if len(images) >= config.reference_max_per_class:
                break
    if not images:
        raise ValueError(f"No CIFAR-10 references found for class {class_id}.")
    return torch.stack(images, dim=0)


def compute_psnr(image_a: torch.Tensor, image_b: torch.Tensor) -> float:
    mse_value = F.mse_loss(image_a, image_b).item()
    if mse_value == 0:
        return float("inf")
    return float(20.0 * math.log10(1.0 / math.sqrt(mse_value)))


def compute_ssim(image_a: torch.Tensor, image_b: torch.Tensor) -> float:
    a = image_a.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    b = image_b.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    return float(structural_similarity(a, b, channel_axis=2, data_range=1.0))


def reference_cache_path(config: ExperimentConfig, class_id: int) -> Path:
    checkpoint_name = "none"
    if config.checkpoint_path:
        checkpoint_name = Path(config.checkpoint_path).stem.replace(" ", "_")
    filename = (
        f"{config.variant}_{config.weight_source}_{checkpoint_name}"
        f"_class{class_id}_{config.reference_split}_{config.reference_max_per_class}.pt"
    )
    return CACHE_DIR / filename


def load_or_compute_reference_embeddings(
    model: ResNetCIFAR,
    references: torch.Tensor,
    config: ExperimentConfig,
    class_id: int,
    batch_size: int = 128,
) -> torch.Tensor:
    cache_path = reference_cache_path(config, class_id)
    if cache_path.exists():
        payload = torch.load(cache_path, map_location="cpu")
        if payload.get("count") == int(references.shape[0]):
            return payload["embeddings"]

    embeddings: List[torch.Tensor] = []
    model.eval()
    with torch.no_grad():
        for start in range(0, references.shape[0], batch_size):
            batch = references[start:start + batch_size].to(config.device)
            embedding = model.forward_features(batch)
            embeddings.append(F.normalize(embedding, dim=1).detach().cpu())
    reference_embeddings = torch.cat(embeddings, dim=0)
    torch.save({"count": int(references.shape[0]), "embeddings": reference_embeddings}, cache_path)
    return reference_embeddings


def find_nearest_reference(model: ResNetCIFAR, reconstruction: torch.Tensor, references: torch.Tensor, config: ExperimentConfig, class_id: int) -> Tuple[torch.Tensor, Dict[str, float]]:
    model.eval()
    reference_embeddings = load_or_compute_reference_embeddings(model, references, config, class_id).to(config.device)
    with torch.no_grad():
        recon_embedding = model.forward_features(reconstruction.to(config.device))
        recon_embedding = F.normalize(recon_embedding, dim=1)
        scores = torch.matmul(reference_embeddings, recon_embedding.squeeze(0))
        best_index = int(torch.argmax(scores).detach().cpu().item())
        best_score = float(scores[best_index].detach().cpu().item())
        best_image = references[best_index:best_index + 1]

    recon_cpu = reconstruction.detach().cpu()
    nearest_cpu = best_image.detach().cpu()
    metrics = {
        "nearest_index_within_class": int(best_index),
        "nearest_embedding_cosine": float(best_score),
        "SSIM": compute_ssim(recon_cpu, nearest_cpu),
        "PSNR": compute_psnr(recon_cpu, nearest_cpu),
        "MSE": float(F.mse_loss(recon_cpu, nearest_cpu).item()),
    }
    return nearest_cpu, metrics


def format_metric(value: Any, digits: int = 4) -> str:
    if value is None:
        return "n/a"
    try:
        if isinstance(value, float) and math.isinf(value):
            return "inf"
        return f"{float(value):.{digits}f}"
    except Exception:
        return str(value)


def comparison_metrics_text(row: Dict[str, Any]) -> str:
    metric_lines = [
        f"Model: {row.get('variant', 'n/a')}",
        f"Objective: {row.get('objective_name', 'phase1_original')}",
        f"Class: {row.get('class_id', 'n/a')} - {row.get('class_name', 'n/a')}",
        f"Seed: {row.get('seed', 'n/a')}",
        "",
        f"SSIM: {format_metric(row.get('SSIM'))}",
        f"PSNR: {format_metric(row.get('PSNR'))}",
        f"MSE: {format_metric(row.get('MSE'))}",
        f"Nearest cosine: {format_metric(row.get('nearest_embedding_cosine'))}",
        f"Target conf: {format_metric(row.get('target_confidence'))}",
        f"Top-k hit: {row.get('topk_contains_target', 'n/a')}",
        f"Runtime: {format_metric(row.get('runtime_seconds'), digits=2)}s",
    ]
    if "candidate_score" in row:
        metric_lines.append(f"Candidate score: {format_metric(row.get('candidate_score'))}")
    return "\n".join(metric_lines)


def save_reconstruction_grid(reconstruction: torch.Tensor, nearest: torch.Tensor, row: Dict[str, Any]) -> Path:
    class_name = CIFAR10_CLASSES[int(row["class_id"])] if int(row["class_id"]) < len(CIFAR10_CLASSES) else str(row["class_id"])
    objective_name = str(row.get("objective_name", "phase1_original")).replace(" ", "_")
    filename = f"{row['variant']}_{objective_name}_class{row['class_id']}_{class_name}_seed{row['seed']}.png"
    path = FIGURE_DIR / filename
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.25]})
    axes[0].set_title("Reconstructed")
    axes[0].imshow(to_pil_image(reconstruction.detach().cpu().squeeze(0)))
    axes[0].axis("off")
    axes[1].set_title("Nearest actual reference")
    axes[1].imshow(to_pil_image(nearest.detach().cpu().squeeze(0)))
    axes[1].axis("off")
    axes[2].axis("off")
    axes[2].text(0.02, 0.98, comparison_metrics_text(row), va="top", ha="left", fontsize=9, family="monospace")
    fig.suptitle("Actual vs Reconstructed with Metrics", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.savefig(path, dpi=180)
    plt.close(fig)
    return path

## Benchmark Runner

Set `RUN_BENCHMARK = True` only when CIFAR-10 data and checkpoints are available. The default keeps notebook execution fast for validation.

In [7]:
def run_one_case(config: ExperimentConfig, class_id: int, seed: int) -> Dict[str, Any]:
    model, provenance = load_model(config, seed=seed)
    reconstruction, recon_metrics = reconstruct_from_weights(model, class_id, seed, config)
    references = load_cifar10_reference_tensors(config, class_id)
    nearest, nearest_metrics = find_nearest_reference(model, reconstruction, references, config, class_id)
    row: Dict[str, Any] = {
        **asdict(config),
        "class_id": int(class_id),
        "class_name": CIFAR10_CLASSES[int(class_id)] if int(class_id) < len(CIFAR10_CLASSES) else str(class_id),
        "seed": int(seed),
        **{key: value for key, value in provenance.items() if key not in ("skipped_layers", "unexpected_layers")},
        **recon_metrics,
        **nearest_metrics,
    }
    row["figure_path"] = str(save_reconstruction_grid(reconstruction, nearest, row))
    return row


def save_results(rows: Sequence[Dict[str, Any]], result_name: str = "phase1_baseline_metrics") -> Tuple[Path, Path]:
    json_path = RESULTS_DIR / f"{result_name}.json"
    csv_path = RESULTS_DIR / f"{result_name}.csv"
    with json_path.open("w", encoding="utf-8") as handle:
        json.dump(list(rows), handle, indent=2, default=str)
    if rows:
        fieldnames = sorted({key for row in rows for key in row.keys()})
        with csv_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    return csv_path, json_path


def run_benchmark(config: ExperimentConfig) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    total = len(config.class_ids) * len(config.seeds)
    progress = tqdm(total=total, desc=f"{config.variant} reconstruction")
    for class_id in config.class_ids:
        for seed in config.seeds:
            rows.append(run_one_case(config, int(class_id), int(seed)))
            progress.update(1)
    progress.close()
    csv_path, json_path = save_results(rows)
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return pd.DataFrame(rows)


def summarize_metrics(results: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["SSIM", "PSNR", "MSE", "nearest_embedding_cosine", "target_confidence", "runtime_seconds"]
    summary = results.groupby(["variant", "weight_source"])[metric_cols].agg(["mean", "std"])
    return summary.reset_index()

## Baseline Comparison

Use this after running the benchmark. Phase 2 is successful only if the improved configuration beats this Phase 1 table on aggregate metrics, not on a single cherry-picked image.

In [8]:
def compare_to_prior_report(results: pd.DataFrame) -> pd.DataFrame:
    current = results.groupby("variant")[["SSIM", "PSNR", "MSE", "nearest_embedding_cosine"]].mean().reset_index()
    current = current.rename(columns={"nearest_embedding_cosine": "cosine"})
    merged = current.merge(PRIOR_REPORTED_BASELINE, on="variant", suffixes=("_phase1", "_prior"))
    for metric in ("SSIM", "PSNR", "MSE", "cosine"):
        merged[f"delta_{metric}"] = merged[f"{metric}_phase1"] - merged[f"{metric}_prior"]
    return merged


PRIOR_REPORTED_BASELINE

,variant,SSIM,cosine,PSNR,MSE
0,resnet18,0.0683,0.421,7.249,0.188
1,resnet34,0.1331,0.410,7.281,0.187
2,resnet50,0.0786,0.405,6.119,0.244
3,resnet101,0.0938,0.761,6.462,0.225
4,resnet152,0.0801,0.507,6.532,0.222


## Phase 2 Objective Registry

Phase 2 keeps the same model-loading and metric harness, but adds research-backed objective variants. The goal is to improve reconstruction quality with a controlled ablation path rather than hand-tuning one image.

Implemented here:
- DeepInversion-style BatchNorm-stat prior across every useful BatchNorm layer.
- Plug-In Inversion style augmentation views for robust optimization.
- Target-class margin loss and optional non-target suppression.
- Direct, sigmoid, and tanh image parameterization.
- Cosine learning-rate schedule and scheduled loss weights.
- Multi-start candidate generation with a deterministic composite selection score.

Deferred intentionally: VGG/perceptual loss. It adds dependency and runtime cost, so it should be added only if the simpler objectives plateau.

In [9]:
@dataclass(frozen=True)
class ObjectiveConfig:
    name: str
    parameterization: str = "direct"
    use_augmentations: bool = False
    multi_crop_count: int = 1
    jitter_pixels: int = 2
    crop_min_scale: float = 0.85
    color_jitter_strength: float = 0.08
    allow_horizontal_flip: bool = True
    margin: float = 1.0
    lambda_embedding: float = 1.0
    lambda_cosine: float = 0.5
    lambda_logit: float = 0.1
    lambda_margin: float = 0.0
    lambda_negative: float = 0.0
    lambda_tv: float = 1e-5
    lambda_l2: float = 1e-4
    lambda_bn: float = 1e-2
    negative_topk: int = 3
    bn_warmup_fraction: float = 0.25
    class_warmup_fraction: float = 0.10
    cosine_lr: bool = True
    multistart_count: int = 1
    low_frequency_init: bool = False


PHASE2_OBJECTIVES: Dict[str, ObjectiveConfig] = {
    "phase1_registered": ObjectiveConfig(
        name="phase1_registered",
        parameterization="direct",
        use_augmentations=False,
        multi_crop_count=1,
        lambda_embedding=1.0,
        lambda_cosine=0.5,
        lambda_logit=0.1,
        lambda_margin=0.0,
        lambda_negative=0.0,
        lambda_tv=1e-5,
        lambda_l2=1e-4,
        lambda_bn=1e-2,
        multistart_count=1,
    ),
    "deepinv_margin": ObjectiveConfig(
        name="deepinv_margin",
        parameterization="direct",
        use_augmentations=False,
        lambda_bn=2e-2,
        lambda_margin=0.20,
        lambda_negative=0.02,
        lambda_tv=2e-5,
        lambda_l2=1e-4,
        multistart_count=1,
    ),
    "plug_in_aug_sigmoid": ObjectiveConfig(
        name="plug_in_aug_sigmoid",
        parameterization="sigmoid",
        use_augmentations=True,
        multi_crop_count=3,
        jitter_pixels=2,
        crop_min_scale=0.82,
        color_jitter_strength=0.06,
        lambda_bn=2.5e-2,
        lambda_margin=0.25,
        lambda_negative=0.04,
        lambda_tv=3e-5,
        lambda_l2=7e-5,
        multistart_count=2,
        low_frequency_init=True,
    ),
    "tanh_multistart": ObjectiveConfig(
        name="tanh_multistart",
        parameterization="tanh",
        use_augmentations=True,
        multi_crop_count=2,
        lambda_bn=2e-2,
        lambda_margin=0.20,
        lambda_negative=0.03,
        multistart_count=2,
        low_frequency_init=True,
    ),
}


def scheduled_weight(base_weight: float, step: int, total_steps: int, warmup_fraction: float) -> float:
    if base_weight == 0:
        return 0.0
    warmup_steps = max(1, int(total_steps * warmup_fraction))
    return float(base_weight * min(1.0, step / warmup_steps))


def initialize_image_parameter(config: ExperimentConfig, objective: ObjectiveConfig, seed: int) -> torch.Tensor:
    set_seed(seed)
    if objective.low_frequency_init:
        small_size = max(4, config.image_size // 4)
        base = torch.rand(1, 3, small_size, small_size, device=config.device)
        base = F.interpolate(base, size=(config.image_size, config.image_size), mode="bilinear", align_corners=False)
    else:
        base = torch.rand(1, 3, config.image_size, config.image_size, device=config.device)
    base = base.mul(0.8).add(0.1).clamp(1e-4, 1.0 - 1e-4)
    if objective.parameterization == "direct":
        parameter = base.detach().clone()
    elif objective.parameterization == "sigmoid":
        parameter = torch.logit(base).detach().clone()
    elif objective.parameterization == "tanh":
        centered = base.mul(2.0).sub(1.0).clamp(-0.999, 0.999)
        parameter = torch.atanh(centered).detach().clone()
    else:
        raise ValueError(f"Unsupported image parameterization: {objective.parameterization}")
    parameter.requires_grad_(True)
    return parameter


def render_image_parameter(parameter: torch.Tensor, objective: ObjectiveConfig) -> torch.Tensor:
    if objective.parameterization == "direct":
        return parameter.clamp(0, 1)
    if objective.parameterization == "sigmoid":
        return torch.sigmoid(parameter)
    if objective.parameterization == "tanh":
        return torch.tanh(parameter).add(1.0).mul(0.5)
    raise ValueError(f"Unsupported image parameterization: {objective.parameterization}")


def project_image_parameter(parameter: torch.Tensor, objective: ObjectiveConfig) -> None:
    if objective.parameterization == "direct":
        with torch.no_grad():
            parameter.clamp_(0, 1)


def random_resized_view(image: torch.Tensor, min_scale: float) -> torch.Tensor:
    _, _, height, width = image.shape
    scale = random.uniform(min_scale, 1.0)
    crop_h = max(4, int(round(height * scale)))
    crop_w = max(4, int(round(width * scale)))
    top = random.randint(0, height - crop_h) if height > crop_h else 0
    left = random.randint(0, width - crop_w) if width > crop_w else 0
    crop = image[:, :, top:top + crop_h, left:left + crop_w]
    return F.interpolate(crop, size=(height, width), mode="bilinear", align_corners=False)


def color_jitter_view(image: torch.Tensor, strength: float) -> torch.Tensor:
    if strength <= 0:
        return image
    brightness = 1.0 + random.uniform(-strength, strength)
    contrast = 1.0 + random.uniform(-strength, strength)
    view = image * brightness
    mean = view.mean(dim=(2, 3), keepdim=True)
    view = (view - mean) * contrast + mean
    return view.clamp(0, 1)


def jitter_view(image: torch.Tensor, max_pixels: int) -> torch.Tensor:
    if max_pixels <= 0:
        return image
    shift_y = random.randint(-max_pixels, max_pixels)
    shift_x = random.randint(-max_pixels, max_pixels)
    return torch.roll(image, shifts=(shift_y, shift_x), dims=(2, 3))


def build_augmented_views(image: torch.Tensor, objective: ObjectiveConfig) -> List[torch.Tensor]:
    views = [image]
    if not objective.use_augmentations:
        return views
    for _ in range(max(0, objective.multi_crop_count - 1)):
        view = random_resized_view(image, objective.crop_min_scale)
        view = jitter_view(view, objective.jitter_pixels)
        if objective.allow_horizontal_flip and random.random() < 0.5:
            view = torch.flip(view, dims=(3,))
        view = color_jitter_view(view, objective.color_jitter_strength)
        views.append(view)
    return views


def class_margin_terms(logits: torch.Tensor, class_id: int, objective: ObjectiveConfig) -> Tuple[torch.Tensor, torch.Tensor]:
    masked = logits.clone()
    masked[:, class_id] = -torch.inf
    competitor = masked.max(dim=1).values
    target = logits[:, class_id]
    margin_loss = F.relu(objective.margin + competitor - target).mean()
    topk = min(objective.negative_topk, logits.shape[1] - 1)
    if topk <= 0:
        negative_loss = torch.zeros((), device=logits.device)
    else:
        negative_logits = torch.topk(masked, k=topk, dim=1).values
        negative_loss = F.softplus(negative_logits).mean()
    return margin_loss, negative_loss


def phase2_objective_loss(
    model: ResNetCIFAR,
    image: torch.Tensor,
    z_target: torch.Tensor,
    class_id: int,
    config: ExperimentConfig,
    objective: ObjectiveConfig,
    step: int,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    views = build_augmented_views(image, objective)
    term_tensors: Dict[str, torch.Tensor] = {
        "embedding_loss": torch.zeros((), device=image.device),
        "cosine_loss": torch.zeros((), device=image.device),
        "logit_loss": torch.zeros((), device=image.device),
        "margin_loss": torch.zeros((), device=image.device),
        "negative_loss": torch.zeros((), device=image.device),
        "bn_loss": torch.zeros((), device=image.device),
    }
    for view in views:
        embedding, logits, bn_loss = forward_with_bn_loss(model, view)
        cosine = F.cosine_similarity(embedding, z_target.unsqueeze(0), dim=1).mean()
        margin_loss, negative_loss = class_margin_terms(logits, class_id, objective)
        term_tensors["embedding_loss"] += F.mse_loss(embedding, z_target.unsqueeze(0))
        term_tensors["cosine_loss"] += 1.0 - cosine
        term_tensors["logit_loss"] += -logits[:, class_id].mean()
        term_tensors["margin_loss"] += margin_loss
        term_tensors["negative_loss"] += negative_loss
        term_tensors["bn_loss"] += bn_loss
    for key in list(term_tensors):
        term_tensors[key] = term_tensors[key] / len(views)
    term_tensors["tv_loss"] = total_variation(image)
    term_tensors["l2_loss"] = image.pow(2).mean()

    weights = {
        "embedding_loss": objective.lambda_embedding,
        "cosine_loss": objective.lambda_cosine,
        "logit_loss": scheduled_weight(objective.lambda_logit, step, config.steps_image, objective.class_warmup_fraction),
        "margin_loss": scheduled_weight(objective.lambda_margin, step, config.steps_image, objective.class_warmup_fraction),
        "negative_loss": scheduled_weight(objective.lambda_negative, step, config.steps_image, objective.class_warmup_fraction),
        "bn_loss": scheduled_weight(objective.lambda_bn, step, config.steps_image, objective.bn_warmup_fraction),
        "tv_loss": objective.lambda_tv,
        "l2_loss": objective.lambda_l2,
    }
    loss = sum(weights[key] * term_tensors[key] for key in weights)
    terms = {key: float(value.detach().cpu().item()) for key, value in term_tensors.items()}
    terms.update({f"weight_{key}": float(value) for key, value in weights.items()})
    terms["objective_loss"] = float(loss.detach().cpu().item())
    return loss, terms


def score_candidate(model: ResNetCIFAR, image: torch.Tensor, z_target: torch.Tensor, class_id: int, config: ExperimentConfig) -> Dict[str, Any]:
    with torch.no_grad():
        embedding, logits, _ = forward_with_bn_loss(model, image)
        probabilities = torch.softmax(logits, dim=1)
        top_values, top_indices = torch.topk(probabilities, k=min(config.top_k, config.num_classes), dim=1)
        cosine = F.cosine_similarity(embedding, z_target.unsqueeze(0), dim=1).mean()
        topk_contains_target = bool(class_id in top_indices[0].detach().cpu().tolist())
        score = 0.50 * float(cosine.item()) + 0.35 * float(probabilities[0, class_id].item()) + 0.15 * float(topk_contains_target)
    return {
        "candidate_score": float(score),
        "embedding_cosine_to_target": float(cosine.detach().cpu().item()),
        "target_confidence": float(probabilities[0, class_id].detach().cpu().item()),
        "top1_class": int(top_indices[0, 0].detach().cpu().item()),
        "top1_confidence": float(top_values[0, 0].detach().cpu().item()),
        "topk_contains_target": topk_contains_target,
    }


def reconstruct_with_objective(
    model: ResNetCIFAR,
    class_id: int,
    seed: int,
    config: ExperimentConfig,
    objective: ObjectiveConfig,
    z_target: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, Dict[str, Any]]:
    set_seed(seed)
    if z_target is None:
        z_target = optimize_target_embedding(model, class_id, config)
    parameter = initialize_image_parameter(config, objective, seed)
    optimizer = optim.Adam([parameter], lr=config.lr_image)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, config.steps_image)) if objective.cosine_lr else None
    start = time.perf_counter()
    last_terms: Dict[str, float] = {}
    for step in range(1, config.steps_image + 1):
        optimizer.zero_grad(set_to_none=True)
        image = render_image_parameter(parameter, objective)
        loss, last_terms = phase2_objective_loss(model, image, z_target, class_id, config, objective, step)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        project_image_parameter(parameter, objective)
    final_image = render_image_parameter(parameter, objective).detach().clamp(0, 1)
    score_metrics = score_candidate(model, final_image, z_target, class_id, config)
    score_metrics.update(last_terms)
    score_metrics.update({
        "runtime_seconds": float(time.perf_counter() - start),
        "objective_name": objective.name,
        "parameterization": objective.parameterization,
        "use_augmentations": objective.use_augmentations,
        "multi_crop_count": objective.multi_crop_count,
        "multistart_count": objective.multistart_count,
    })
    return final_image, score_metrics


def reconstruct_with_multistart(
    model: ResNetCIFAR,
    class_id: int,
    seed: int,
    config: ExperimentConfig,
    objective: ObjectiveConfig,
) -> Tuple[torch.Tensor, Dict[str, Any]]:
    best_image: Optional[torch.Tensor] = None
    best_metrics: Optional[Dict[str, Any]] = None
    set_seed(seed)
    z_target = optimize_target_embedding(model, class_id, config)
    for start_idx in range(max(1, objective.multistart_count)):
        candidate_seed = seed + 1009 * start_idx
        image, metrics = reconstruct_with_objective(model, class_id, candidate_seed, config, objective, z_target=z_target)
        metrics["candidate_seed"] = int(candidate_seed)
        metrics["candidate_rank"] = int(start_idx)
        if best_metrics is None or metrics["candidate_score"] > best_metrics["candidate_score"]:
            best_image = image
            best_metrics = metrics
    assert best_image is not None and best_metrics is not None
    return best_image, best_metrics


## Phase 2 Ablation Runner

Run the Phase 1 baseline first. Then run this ablation on the same class/seed subset. The key table is grouped by objective name, so improvement claims can be made from aggregate SSIM, PSNR, MSE, cosine, confidence, and runtime.

In [10]:
def run_one_case_with_objective(config: ExperimentConfig, class_id: int, seed: int, objective: ObjectiveConfig) -> Dict[str, Any]:
    model, provenance = load_model(config, seed=seed)
    reconstruction, recon_metrics = reconstruct_with_multistart(model, class_id, seed, config, objective)
    references = load_cifar10_reference_tensors(config, class_id)
    nearest, nearest_metrics = find_nearest_reference(model, reconstruction, references, config, class_id)
    row: Dict[str, Any] = {
        **asdict(config),
        "class_id": int(class_id),
        "class_name": CIFAR10_CLASSES[int(class_id)] if int(class_id) < len(CIFAR10_CLASSES) else str(class_id),
        "seed": int(seed),
        **{key: value for key, value in provenance.items() if key not in ("skipped_layers", "unexpected_layers")},
        **recon_metrics,
        **nearest_metrics,
    }
    row["figure_path"] = str(save_reconstruction_grid(reconstruction, nearest, row))
    return row


def run_phase2_ablation(config: ExperimentConfig, objective_names: Sequence[str]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    total = len(objective_names) * len(config.class_ids) * len(config.seeds)
    progress = tqdm(total=total, desc=f"{config.variant} phase2 ablation")
    for objective_name in objective_names:
        objective = PHASE2_OBJECTIVES[objective_name]
        for class_id in config.class_ids:
            for seed in config.seeds:
                rows.append(run_one_case_with_objective(config, int(class_id), int(seed), objective))
                progress.update(1)
    progress.close()
    csv_path, json_path = save_results(rows, result_name="phase2_objective_ablation_metrics")
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return pd.DataFrame(rows)


def summarize_phase2_ablation(results: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["SSIM", "PSNR", "MSE", "nearest_embedding_cosine", "target_confidence", "candidate_score", "runtime_seconds"]
    summary = results.groupby(["variant", "weight_source", "objective_name"])[metric_cols].agg(["mean", "std"])
    return summary.reset_index()


def compare_phase1_and_phase2(phase1_results: pd.DataFrame, phase2_results: pd.DataFrame) -> pd.DataFrame:
    phase1 = phase1_results.copy()
    if "objective_name" not in phase1.columns:
        phase1["objective_name"] = "phase1_original"
    combined = pd.concat([phase1, phase2_results], ignore_index=True, sort=False)
    return summarize_phase2_ablation(combined)


RUN_PHASE2_ABLATION = False
phase2_objective_names = ("phase1_registered", "deepinv_margin", "plug_in_aug_sigmoid")
phase2_config = replace(
    BASE_CONFIG,
    variant="resnet18",
    weight_source="local_cifar10",
    checkpoint_path=str(LOCAL_CHECKPOINTS["resnet18"]),
    class_ids=(0, 1, 8),
    seeds=(7, 21, 42),
    reference_max_per_class=500,
    download_cifar10=False,
)

if RUN_PHASE2_ABLATION:
    phase2_results = run_phase2_ablation(phase2_config, phase2_objective_names)
    display(phase2_results)
    display(summarize_phase2_ablation(phase2_results))
else:
    print("Phase 2 ablation skipped. Set RUN_PHASE2_ABLATION=True after Phase 1 baseline is reviewed.")
    print("Available objectives:", list(PHASE2_OBJECTIVES))


Phase 2 ablation skipped. Set RUN_PHASE2_ABLATION=True after Phase 1 baseline is reviewed.
Available objectives: ['phase1_registered', 'deepinv_margin', 'plug_in_aug_sigmoid', 'tanh_multistart']


## Phase 3 Cross-Model Generalization

Phase 3 checks whether the best Phase 2 objective generalizes across approved ResNet variants. It stays CIFAR-10 first because all metrics are comparable at the same 32x32 resolution. ImageNet and decoder comparison remain opt-in because they require different data/checkpoints and can confuse the core metric claim.

Use this after Phase 2 ablation identifies a candidate objective. If only ResNet-18 is available locally, the runner reports missing variants instead of silently failing.

In [11]:
RESNET_DEPTHS = {
    "resnet18": 18,
    "resnet34": 34,
    "resnet50": 50,
    "resnet101": 101,
    "resnet152": 152,
}

PHASE3_APPROVED_CIFAR_VARIANTS = ("resnet18", "resnet34", "resnet50", "resnet101", "resnet152")


def checkpoint_inventory(variant_names: Sequence[str] = PHASE3_APPROVED_CIFAR_VARIANTS) -> pd.DataFrame:
    rows = []
    for variant in variant_names:
        checkpoint_path = LOCAL_CHECKPOINTS.get(variant)
        exists = bool(checkpoint_path is not None and Path(checkpoint_path).exists())
        rows.append({
            "variant": variant,
            "depth": RESNET_DEPTHS.get(variant),
            "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else None,
            "checkpoint_available": exists,
            "status": "ready" if exists else "missing_checkpoint",
        })
    return pd.DataFrame(rows)


def mnist_checkpoint_inventory() -> pd.DataFrame:
    rows = []
    for checkpoint_name, checkpoint_path in MNIST_CHECKPOINTS.items():
        exists = Path(checkpoint_path).exists()
        rows.append({
            "checkpoint_name": checkpoint_name,
            "checkpoint_path": str(checkpoint_path),
            "checkpoint_available": bool(exists),
            "status": "available_for_future_mnist_eval" if exists else "missing_checkpoint",
        })
    return pd.DataFrame(rows)


def phase3_variant_config(base_config: ExperimentConfig, variant: str) -> ExperimentConfig:
    checkpoint_path = LOCAL_CHECKPOINTS.get(variant)
    return replace(
        base_config,
        variant=variant,
        weight_source="local_cifar10",
        checkpoint_path=str(checkpoint_path) if checkpoint_path is not None else None,
        image_size=32,
        num_classes=10,
    )


def add_normalized_metric_fields(results: pd.DataFrame) -> pd.DataFrame:
    enriched = results.copy()
    if enriched.empty:
        return enriched
    enriched["dataset_name"] = "CIFAR-10"
    enriched["resolution"] = enriched["image_size"].astype(str) + "x" + enriched["image_size"].astype(str)
    enriched["pixel_count"] = 3 * enriched["image_size"].astype(int) * enriched["image_size"].astype(int)
    enriched["depth"] = enriched["variant"].map(RESNET_DEPTHS)
    enriched["MSE_normalized"] = enriched["MSE"]
    enriched["PSNR_normalized"] = enriched["PSNR"]
    return enriched


def run_phase3_cross_model_evaluation(
    base_config: ExperimentConfig,
    objective_name: str,
    variant_names: Sequence[str] = PHASE3_APPROVED_CIFAR_VARIANTS,
    include_missing_rows: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if objective_name not in PHASE2_OBJECTIVES:
        raise ValueError(f"Unknown objective: {objective_name}")
    inventory = checkpoint_inventory(variant_names)
    objective = PHASE2_OBJECTIVES[objective_name]
    rows: List[Dict[str, Any]] = []
    available_variants = inventory[inventory["checkpoint_available"]]["variant"].tolist()
    total = len(available_variants) * len(base_config.class_ids) * len(base_config.seeds)
    progress = tqdm(total=total, desc=f"phase3 {objective_name}")
    for _, inventory_row in inventory.iterrows():
        variant = str(inventory_row["variant"])
        if not bool(inventory_row["checkpoint_available"]):
            if include_missing_rows:
                rows.append({
                    "variant": variant,
                    "depth": RESNET_DEPTHS.get(variant),
                    "objective_name": objective_name,
                    "status": "skipped_missing_checkpoint",
                    "checkpoint_path": inventory_row["checkpoint_path"],
                    "weight_source": "local_cifar10",
                    "dataset_name": "CIFAR-10",
                    "image_size": int(base_config.image_size),
                    "num_classes": int(base_config.num_classes),
                })
            continue
        config = phase3_variant_config(base_config, variant)
        for class_id in config.class_ids:
            for seed in config.seeds:
                row = run_one_case_with_objective(config, int(class_id), int(seed), objective)
                row["status"] = "completed"
                rows.append(row)
                progress.update(1)
    progress.close()
    results = add_normalized_metric_fields(pd.DataFrame(rows))
    csv_path, json_path = save_results(results.to_dict("records"), result_name="phase3_cross_model_metrics")
    print("Saved:", csv_path)
    print("Saved:", json_path)
    return results, inventory


def summarize_phase3_cross_model(results: pd.DataFrame) -> pd.DataFrame:
    completed = results[results["status"] == "completed"].copy() if "status" in results.columns else results.copy()
    if completed.empty:
        return pd.DataFrame()
    metric_cols = ["SSIM", "PSNR", "MSE", "nearest_embedding_cosine", "target_confidence", "candidate_score", "runtime_seconds"]
    summary = completed.groupby(["variant", "depth", "objective_name", "dataset_name", "resolution"])[metric_cols].agg(["mean", "std"])
    return summary.reset_index()


def phase3_class_success_failure(results: pd.DataFrame) -> pd.DataFrame:
    completed = results[results["status"] == "completed"].copy() if "status" in results.columns else results.copy()
    if completed.empty:
        return pd.DataFrame()
    completed["topk_success"] = completed["topk_contains_target"].astype(bool)
    grouped = completed.groupby(["variant", "objective_name", "class_id", "class_name"]).agg(
        runs=("seed", "count"),
        topk_success_rate=("topk_success", "mean"),
        mean_ssim=("SSIM", "mean"),
        mean_psnr=("PSNR", "mean"),
        mean_mse=("MSE", "mean"),
        mean_candidate_score=("candidate_score", "mean"),
    )
    return grouped.reset_index().sort_values(["variant", "mean_candidate_score"], ascending=[True, False])


def choose_phase3_serving_candidate(results: pd.DataFrame) -> Dict[str, Any]:
    completed = results[results["status"] == "completed"].copy() if "status" in results.columns else results.copy()
    if completed.empty:
        return {"status": "no_completed_runs"}
    grouped = completed.groupby(["variant", "objective_name", "checkpoint_path"]).agg(
        mean_candidate_score=("candidate_score", "mean"),
        mean_ssim=("SSIM", "mean"),
        mean_psnr=("PSNR", "mean"),
        mean_mse=("MSE", "mean"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        runs=("seed", "count"),
    ).reset_index()
    grouped = grouped.sort_values(
        ["mean_candidate_score", "mean_ssim", "mean_psnr", "mean_mse"],
        ascending=[False, False, False, True],
    )
    best = grouped.iloc[0].to_dict()
    best.update({
        "status": "candidate_selected",
        "dataset_name": "CIFAR-10",
        "class_ids": list(completed["class_id"].dropna().astype(int).unique()),
        "seeds": list(completed["seed"].dropna().astype(int).unique()),
        "approval_required_before_serving": True,
    })
    return best


def freeze_phase3_candidate(results: pd.DataFrame, output_path: Path = RESULTS_DIR / "phase3_frozen_config.json") -> Dict[str, Any]:
    candidate = choose_phase3_serving_candidate(results)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(candidate, handle, indent=2, default=str)
    print("Saved frozen candidate summary:", output_path)
    return candidate


def decoder_comparison_inventory() -> Dict[str, Any]:
    decoder_path = MODEL_DIR / "decoder_trained.pth"
    return {
        "decoder_path": str(decoder_path),
        "decoder_checkpoint_available": decoder_path.exists(),
        "status": "ready_for_approved_comparison" if decoder_path.exists() else "deferred_missing_decoder_checkpoint",
        "note": "No new decoder training is part of Phase 3 unless explicitly approved.",
    }


RUN_PHASE3_CROSS_MODEL = False
PHASE3_OBJECTIVE_NAME = "plug_in_aug_sigmoid"
phase3_config = replace(
    BASE_CONFIG,
    class_ids=(0, 1, 8),
    seeds=(7, 21, 42),
    reference_max_per_class=500,
    download_cifar10=False,
)

display(checkpoint_inventory())
display(mnist_checkpoint_inventory())
print("Decoder comparison:", decoder_comparison_inventory())

if RUN_PHASE3_CROSS_MODEL:
    phase3_results, phase3_inventory = run_phase3_cross_model_evaluation(phase3_config, PHASE3_OBJECTIVE_NAME)
    display(phase3_inventory)
    display(phase3_results)
    display(summarize_phase3_cross_model(phase3_results))
    display(phase3_class_success_failure(phase3_results))
    display(pd.DataFrame([choose_phase3_serving_candidate(phase3_results)]))
else:
    print("Phase 3 cross-model evaluation skipped. Set RUN_PHASE3_CROSS_MODEL=True after Phase 2 objective review.")


,variant,depth,checkpoint_path,checkpoint_available,status
0,resnet18,18,D:\IE_643_Project\pre_trained_model_weights\re...,True,ready
1,resnet34,34,D:\IE_643_Project\pre_trained_model_weights\re...,True,ready
2,resnet50,50,D:\IE_643_Project\pre_trained_model_weights\re...,True,ready
3,resnet101,101,D:\IE_643_Project\pre_trained_model_weights\re...,True,ready
4,resnet152,152,D:\IE_643_Project\pre_trained_model_weights\re...,True,ready


,checkpoint_name,checkpoint_path,checkpoint_available,status
0,resnet18_best,D:\IE_643_Project\pre_trained_model_weights\re...,True,available_for_future_mnist_eval
1,resnet18_final,D:\IE_643_Project\pre_trained_model_weights\re...,True,available_for_future_mnist_eval
2,resnet34_final,D:\IE_643_Project\pre_trained_model_weights\re...,True,available_for_future_mnist_eval
3,resnet152_final,D:\IE_643_Project\pre_trained_model_weights\re...,True,available_for_future_mnist_eval


Decoder comparison: {'decoder_path': 'D:\\IE_643_Project\\Model_street_final\\decoder_trained.pth', 'decoder_checkpoint_available': False, 'status': 'deferred_missing_decoder_checkpoint', 'note': 'No new decoder training is part of Phase 3 unless explicitly approved.'}
Phase 3 cross-model evaluation skipped. Set RUN_PHASE3_CROSS_MODEL=True after Phase 2 objective review.


## Notebook Smoke Tests

These tests avoid dataset downloads and heavy training. They prove that the notebook structure, model builder, deterministic seed path, metric math, baseline reconstruction loop, and Phase 2 objective loop are wired correctly.

In [12]:
def run_smoke_tests() -> None:
    smoke_config = ExperimentConfig(
        variant="resnet18",
        weight_source="random",
        checkpoint_path=None,
        class_ids=(0,),
        seeds=(123,),
        steps_z=2,
        steps_image=2,
        lambda_bn=0.0,
        device=str(DEVICE),
    )
    model_a, provenance_a = load_model(smoke_config, seed=123)
    model_b, _ = load_model(smoke_config, seed=123)
    assert provenance_a["weight_source"] == "random"
    test_image = torch.rand(1, 3, smoke_config.image_size, smoke_config.image_size, device=smoke_config.device)
    logits = model_a(test_image)
    assert tuple(logits.shape) == (1, smoke_config.num_classes)
    first_weight_a = next(model_a.parameters()).detach().cpu()
    first_weight_b = next(model_b.parameters()).detach().cpu()
    assert torch.allclose(first_weight_a, first_weight_b), "seeded model initialization must be deterministic"
    same = torch.zeros(1, 3, smoke_config.image_size, smoke_config.image_size)
    assert math.isinf(compute_psnr(same, same))
    reconstruction, metrics = reconstruct_from_weights(model_a, class_id=0, seed=123, config=smoke_config)
    assert tuple(reconstruction.shape) == (1, 3, smoke_config.image_size, smoke_config.image_size)
    assert 0.0 <= float(reconstruction.min()) <= float(reconstruction.max()) <= 1.0
    assert "target_confidence" in metrics
    phase2_smoke_objective = replace(PHASE2_OBJECTIVES["deepinv_margin"], name="phase2_smoke", multistart_count=1, use_augmentations=True, multi_crop_count=2, lambda_bn=0.0)
    reconstruction_v2, metrics_v2 = reconstruct_with_multistart(model_a, class_id=0, seed=123, config=smoke_config, objective=phase2_smoke_objective)
    assert tuple(reconstruction_v2.shape) == (1, 3, smoke_config.image_size, smoke_config.image_size)
    assert metrics_v2["objective_name"] == "phase2_smoke"
    assert "candidate_score" in metrics_v2
    sample_metrics_panel = comparison_metrics_text({"variant": "resnet18", "class_id": 0, "class_name": "airplane", "seed": 123, "SSIM": 0.1, "PSNR": 8.0, "MSE": 0.2})
    assert "SSIM" in sample_metrics_panel and "PSNR" in sample_metrics_panel and "MSE" in sample_metrics_panel
    phase3_inventory = checkpoint_inventory(("resnet18",))
    assert "checkpoint_available" in phase3_inventory.columns
    mnist_inventory = mnist_checkpoint_inventory()
    assert "checkpoint_name" in mnist_inventory.columns
    skipped_summary = summarize_phase3_cross_model(pd.DataFrame([{"variant": "resnet34", "status": "skipped_missing_checkpoint"}]))
    assert skipped_summary.empty
    assert "decoder_checkpoint_available" in decoder_comparison_inventory()
    print("Smoke tests passed.")


RUN_SMOKE_TESTS = True
if RUN_SMOKE_TESTS:
    run_smoke_tests()

Smoke tests passed.


## Run Phase 1 Baseline

Before running this cell:
- Confirm the checkpoint path exists.
- Set `download_cifar10=True` only if CIFAR-10 is not already under `data/`.
- Start with ResNet-18, three classes, and three seeds. Increase scope after reviewing runtime and stability.

In [13]:
RUN_BENCHMARK = False

phase1_config = replace(
    BASE_CONFIG,
    variant="resnet18",
    weight_source="local_cifar10",
    checkpoint_path=str(LOCAL_CHECKPOINTS["resnet18"]),
    class_ids=(0, 1, 8),
    seeds=(7, 21, 42),
    reference_max_per_class=500,
    download_cifar10=False,
)

if RUN_BENCHMARK:
    phase1_results = run_benchmark(phase1_config)
    display(phase1_results)
    display(summarize_metrics(phase1_results))
    display(compare_to_prior_report(phase1_results))
else:
    print("Benchmark skipped. Set RUN_BENCHMARK=True after confirming data/checkpoints.")
    print("Configured checkpoint:", phase1_config.checkpoint_path)
    print("Checkpoint exists:", Path(phase1_config.checkpoint_path).exists())

Benchmark skipped. Set RUN_BENCHMARK=True after confirming data/checkpoints.
Configured checkpoint: D:\IE_643_Project\pre_trained_model_weights\resnet18_cifar10_trained.pth
Checkpoint exists: True
